# PathSnap — full rerun on Google Colab (GPU)

Recomputes **everything** after the label and feature changes (heading-aware label alignment,
two direction features, heading-aware neighbour features):

| Stage | What | Rough time on free Colab |
|---|---|---|
| 1 | preprocessing: inventory, cleaning, labels, candidate sweep, features, maps | 5-10 min |
| 2 | train + evaluate 7 models (XGBoost on GPU, the rest on 2 CPU cores) | 1.5-2.5 h |
| 3 | Bayesian search, 50 Optuna trials on the GPU + refit of the best | 20-40 min |
| 4 | comparison plots, HPO plots, error analysis | 2 min |
| 5 | execute notebooks 01-04 | 5-10 min |

Times are estimates. Nothing is uploaded anywhere except your own Colab session (and Drive if you turn it on).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Upload `pathsnap_rerun.zip` (from `python -m pathsnap.colab_bundle --fresh`) in the **Files** panel,
   or let the upload cell ask for it.
3. *Runtime → Run all*. Keep the tab open (free Colab disconnects idle sessions).

**If the session disconnects**: a checkpoint zip is written after every stage and model
(`/content/pathsnap_checkpoint.zip`, and on Drive if `USE_DRIVE = True`). Download it, start a new
session, upload both zips, set `RESUME_ZIP` to the checkpoint's name and run all again: finished stages
and models are skipped, and the Optuna study resumes where it stopped.

GPU and CPU XGBoost results are close but not bit-identical; the report should say XGBoost was trained on a T4 GPU.

## 1. Settings

In [ ]:
BUNDLE      = "pathsnap_rerun.zip"
RESUME_ZIP  = ""            # e.g. "pathsnap_checkpoint.zip" to continue a disconnected run
USE_DRIVE   = False         # True: also read zips from / save checkpoints to Google Drive
DRIVE_DIR   = "/content/drive/MyDrive/PathSnap"
MODELS      = ["logreg", "knn", "rf", "lgbm", "mlp", "xgb", "dtree"]
N_TRIALS    = 50
RUN_NOTEBOOKS = True

## 2. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, multiprocessing
print("CPU cores:", multiprocessing.cpu_count())

## 3. Load the bundle (and a checkpoint, if resuming)

In [ ]:
import pathlib, zipfile
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

def find(name):
    for p in [pathlib.Path("/content") / name, pathlib.Path(DRIVE_DIR) / name]:
        if p.exists():
            return p
    from google.colab import files
    print(f"{name} not found - choose it in the dialog")
    up = files.upload()
    return pathlib.Path("/content") / next(iter(up))

bundle = find(BUNDLE)
WORK = pathlib.Path("/content/work")
if not (WORK / "pathsnap").exists():
    with zipfile.ZipFile(bundle) as z:
        z.extractall(WORK)
REPO_DIR, DATA_DIR = WORK / "pathsnap", WORK / "Student folder"
assert (REPO_DIR / "pathsnap" / "config.py").exists() and (DATA_DIR / "train").is_dir(), "unexpected zip layout"
if RESUME_ZIP:
    with zipfile.ZipFile(find(RESUME_ZIP)) as z:
        z.extractall(REPO_DIR)
    print("resumed from", RESUME_ZIP)
%cd {REPO_DIR}

## 4. Install dependencies (full `xgboost` wheel with CUDA instead of `xgboost-cpu`)

In [ ]:
import re, subprocess
reqs = [l.strip() for l in open("requirements.txt") if l.strip() and not l.startswith("#")]
xgb_version = next(l.split("==")[1] for l in reqs if l.startswith("xgboost-cpu"))
others = [l for l in reqs if not l.startswith("xgboost-cpu")]
open("/tmp/req_colab.txt", "w").write("\n".join(others + [f"xgboost=={xgb_version}"]))
r = subprocess.run(["pip", "install", "-q", "-r", "/tmp/req_colab.txt"], capture_output=True, text=True)
if r.returncode != 0:
    print("pinned install failed, installing unpinned:", r.stderr[-500:])
    names = [re.split(r"[=<>]", l)[0] for l in others] + [f"xgboost=={xgb_version}"]
    subprocess.run(["pip", "install", "-q", *names], check=True)
import xgboost, pandas, sklearn
print("xgboost", xgboost.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)

## 5. Environment and checkpoint helper

In [ ]:
import sys, time, shutil
os.environ["PATHSNAP_DATA"] = str(DATA_DIR)
os.environ["PATHSNAP_JOBS"] = str(multiprocessing.cpu_count())
os.environ["PATHSNAP_XGB_DEVICE"] = "cuda"          # XGBoost (comparison + HPO) on the GPU
sys.path.insert(0, str(REPO_DIR))

def checkpoint(include_rf=False, name="pathsnap_checkpoint.zip"):
    """Zip results, notebooks and model weights (paths relative to the repo root)."""
    out = pathlib.Path("/content") / name
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for folder in ["results", "artifacts", "notebooks"]:
            for f in pathlib.Path(folder).rglob("*"):
                if f.is_file() and (include_rf or f.name != "rf.joblib"):
                    z.write(f, f)
    if USE_DRIVE:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        shutil.copy(out, pathlib.Path(DRIVE_DIR) / name)
    print(f"checkpoint: {out} ({out.stat().st_size / 1e6:.1f} MB)")

def run(*args):
    t0 = time.perf_counter()
    subprocess.run([sys.executable, "-m", *args], check=True)
    print(f"--- {' '.join(args)}: {time.perf_counter() - t0:.0f} s")

!python -m pathsnap.reproduce --check

## Stage 1 — preprocessing

Always rebuilt (fast), because labels and features changed.

In [ ]:
from pathsnap.reproduce import run_preprocessing
run_preprocessing()
checkpoint()

## Stage 2 — train and evaluate the models

Same harness as before (5-fold GroupKFold on train, then student test). Finished models are skipped on resume.

In [ ]:
from pathsnap.reproduce import models_ready
for m in MODELS:
    if m not in models_ready([m]):
        print(f"skip {m} (already trained)")
        continue
    run("pathsnap.models.train", m)
    checkpoint()
run("pathsnap.models.train", "--rebuild-comparison")
import pandas as pd
pd.read_csv("results/models/comparison.csv")[["model", "cv_macro_f1", "cv_top1", "test_top1",
                                              "test_row_accuracy", "test_macro_f1", "test_log_loss"]].round(4)

## Stage 3 — Bayesian search (GPU) and refit

The study is saved after every trial; on resume it continues from the checkpoint's `xgb_study.db`.

In [ ]:
run("pathsnap.hpo.xgb_optuna", "--trials", str(N_TRIALS))
checkpoint()
run("pathsnap.hpo.xgb_optuna", "--refit")
checkpoint()
pd.read_csv("results/hpo/xgb_tuned_vs_untuned.csv").round(4)

## Stage 4 — plots and error analysis

In [ ]:
run("pathsnap.models.train", "--rebuild-comparison")
run("pathsnap.viz.model_plots")
run("pathsnap.viz.hpo_plots")
run("pathsnap.models.error_analysis")
run("pathsnap.models.error_analysis", "xgb")
pd.read_csv("results/models/error_breakdown_final_model.csv")

## Stage 5 — execute notebooks 01-04

In [ ]:
if RUN_NOTEBOOKS:
    from pathsnap.reproduce import run_notebooks
    run_notebooks()
checkpoint()

## Download everything

Unzip this inside the repo on the laptop (it replaces `results/`, `artifacts/` and `notebooks/0*.ipynb`), then commit.

In [ ]:
checkpoint(include_rf=True, name="pathsnap_rerun_outputs.zip")
from google.colab import files
files.download("/content/pathsnap_rerun_outputs.zip")